In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from Log_Extractor import LogExtractor
import os

### 어떤 펌프에 대해서?

In [2]:
# ==========================================
# ⚙️ 1. 설정 및 매개변수 (여기만 바꾸면 14개 펌프 무한 확장 가능!)
# ==========================================
PUMP_ID = 'P1' # 나중에 P2, P3 등으로 변경
TANK_ID = 'P1' # 나중에 T2, T3 등으로 변경
MODEL_SAVE_DIR = './models' # 모델과 스케일러를 저장할 폴더
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

In [3]:
def prepare_pump_features(raw_df, pump_id, tank_id):
    """
    raw_df를 받아 해당 펌프(pump_id)의 피처를 생성하여 model_df를 반환합니다.
    """
    df = raw_df.ffill().fillna(0).copy()
    df_reset = df.reset_index() 
    
    # 동적 태그 이름 생성 (P1, P2 등 자동 대응)
    tag_SV = f'g_s_SV_{pump_id}'
    tag_Ana = f'Ana_Out_{pump_id}'
    tag_PT = f'Scale_Out___PT_{pump_id}'
    tag_FT = f'Scale_Out___FT_{pump_id}'
    tag_Temp = f'TK_Temp_PV_{tank_id}'

    # 1. 블록 나누기
    df_reset['Wagon_Block_ID'] = (df_reset['전진_Actual_Wagon_Num'].diff() != 0).cumsum()

    # 2. 요약 및 노이즈 필터링
    block_summary = df_reset.groupby('Wagon_Block_ID').agg(
        Max_SV=(tag_SV, 'max'),
        Duration=('Time', lambda x: x.max() - x.min()) 
    )
    valid_blocks = block_summary[
        (block_summary['Duration'] < pd.Timedelta(minutes=5)) &
        (block_summary['Max_SV'] > 10)
    ].copy()

    valid_blocks['Prev_SV'] = valid_blocks['Max_SV'].shift(1)

    # 3. 진짜 샷 추출
    valid_block_ids = valid_blocks.index
    df_shots = df_reset[
        (df_reset['Wagon_Block_ID'].isin(valid_block_ids)) & 
        (df_reset['Emergency_Sw'] == 1)
    ].copy()
    df_shots = df_shots.merge(valid_blocks[['Prev_SV']], left_on='Wagon_Block_ID', right_index=True, how='left')
    df_shots = df_shots.sort_index()

    # 4. 실시간 피처 엔지니어링
    model_df = df_shots.copy()
    model_df['Tick_Index'] = model_df.groupby('Wagon_Block_ID').cumcount()
    model_df['Phase_Start'] = np.where(model_df['Tick_Index'] < 2, 1.0, 0.0)
    model_df['Phase_Steady'] = np.where(model_df['Tick_Index'] >= 2, 1.0, 0.0)

    model_df['Instant_FT_Error'] = model_df[tag_SV] - model_df[tag_FT]
    model_df['Instant_FT_Error_Rate'] = np.where(
        model_df[tag_SV] > 0, 
        (model_df['Instant_FT_Error'] / model_df[tag_SV]) * 100, 
        0
    )
    model_df['Cum_FT_Error'] = model_df.groupby('Wagon_Block_ID')['Instant_FT_Error'].cumsum()

    model_df['Prev_SV_Diff'] = model_df[tag_SV] - model_df['Prev_SV'].fillna(0)
    model_df['Rolling_PT_Max_3'] = model_df.groupby('Wagon_Block_ID')[tag_PT].rolling(window=3, min_periods=1).max().reset_index(0, drop=True)
    model_df['Rolling_PT_Diff_3'] = model_df.groupby('Wagon_Block_ID')[tag_PT].diff(periods=2).fillna(0)

    model_df['Instant_SV_Diff'] = model_df[tag_SV].diff().fillna(0)
    model_df['Phase_Transition'] = np.where(model_df['Instant_SV_Diff'] != 0, 1.0, 0.0)

    # 5. 최종 컬럼 정리 (메타 정보 2개 제외하고 학습에 들어갈 피처들)
    # *주의: 모델 텐서 변환 시 Wagon_Block_ID와 Tick_Index는 빼야 합니다.
    feature_cols = [
        tag_SV, 'Prev_SV', 'Prev_SV_Diff', 
        tag_Ana, tag_Temp,               
        tag_PT, 'Rolling_PT_Max_3', 'Rolling_PT_Diff_3', 
        tag_FT, 'Instant_FT_Error_Rate', 'Cum_FT_Error',  
        'Phase_Start', 'Phase_Steady', 'Phase_Transition'             
    ]
    
    # 결측치 최종 정리 (Shift나 Rolling에서 발생한 NaN 처리)
    model_df = model_df.fillna(0)
    
    return model_df[feature_cols], feature_cols

print("✅ 동적 피처 생성 함수 준비 완료!")

✅ 동적 피처 생성 함수 준비 완료!


## 받아오는것은 직접해라.

In [4]:
# 한글 폰트 설정
plt.rc('font', family='AppleGothic') 
plt.rcParams['axes.unicode_minus'] = False

# 1. 추출기 가동 (어제 하루치 데이터만 가져오기)
# extractor는 이미 위에서 선언했다고 가정합니다.
target_tags = [
    'Emergency_Sw', '전진_Actual_Wagon_Num', 
    f'g_s_SV_{PUMP_ID}', f'Ana_Out_{PUMP_ID}', 
    f'Scale_Out___PT_{PUMP_ID}', f'Scale_Out___FT_{PUMP_ID}', 
    f'TK_Temp_PV_{TANK_ID}', f'TK_Level_PV_{TANK_ID}'
]
extractor = LogExtractor()

🔌 [Extractor] InfluxDB 분석용 추출기 연결 완료!


# 데이터 불러오기

In [7]:
from datetime import datetime, timedelta

# ==========================================
# 📅 1. 날짜 범위 설정 (최근 7일)
# ==========================================
end_date = datetime.now() - timedelta(days=1)  # 어제까지가 학습 데이터
start_date = end_date - timedelta(days=7)      # 일주일 전부터

# 데이터를 담을 리스트
train_chunks = []

print(f"🔍 {PUMP_ID} 학습 데이터 추출 시작 ({start_date.date()} ~ {end_date.date()})")

# ==========================================
# 🔄 2. 하루 단위 루프 돌리기
# ==========================================
current_date = start_date
while current_date <= end_date:
    # InfluxDB용 시간 포맷팅 (ISO 8601)
    day_start = current_date.strftime('%Y-%m-%dT00:00:00Z')
    day_end = (current_date + timedelta(days=1)).strftime('%Y-%m-%dT00:00:00Z')
    
    print(f"   📅 {current_date.date()} 데이터 요청 중...")
    
    try:
        # 하루치씩 쏙쏙 뽑아오기
        day_raw = extractor.get_data(start_time=day_start, end_time=day_end, target_tags=target_tags)
        
        if not day_raw.empty:
            train_chunks.append(day_raw)
            print(f"      ✅ {len(day_raw)}행 수집 완료")
        else:
            print(f"      ⚠️ 데이터가 없습니다.")
            
    except Exception as e:
        print(f"      ❌ 오류 발생: {e}")
    
    current_date += timedelta(days=1)

# ==========================================
# 🧩 3. 데이터 병합 및 피처 생성
# ==========================================
if train_chunks:
    raw_train = pd.concat(train_chunks)
    print(f"\n✅ 전체 데이터 병합 완료! 총 {len(raw_train)}행")
    
    # 기존 피처 생성 함수 태우기
    train_df, feature_cols = prepare_pump_features(raw_train, PUMP_ID, TANK_ID)
else:
    print("❌ 수집된 데이터가 없어 학습을 진행할 수 없습니다.")

# Validation(오늘치)은 상대적으로 작으므로 기존처럼 한 번에 가져옵니다.
raw_valid = extractor.get_data(start_time="-1d", end_time="now()", target_tags=target_tags)
valid_df, _ = prepare_pump_features(raw_valid, PUMP_ID, TANK_ID)

🔍 P1 학습 데이터 추출 시작 (2026-04-01 ~ 2026-04-08)
   📅 2026-04-01 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-01T00:00:00Z ~ 2026-04-02T00:00:00Z)
✅ 추출 완료! 총 0행, 0개 컬럼 확보.
      ⚠️ 데이터가 없습니다.
   📅 2026-04-02 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-02T00:00:00Z ~ 2026-04-03T00:00:00Z)
✅ 추출 완료! 총 0행, 0개 컬럼 확보.
      ⚠️ 데이터가 없습니다.
   📅 2026-04-03 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-03T00:00:00Z ~ 2026-04-04T00:00:00Z)
✅ 추출 완료! 총 24785행, 11개 컬럼 확보.
      ✅ 24785행 수집 완료
   📅 2026-04-04 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-04T00:00:00Z ~ 2026-04-05T00:00:00Z)
✅ 추출 완료! 총 20132행, 11개 컬럼 확보.
      ✅ 20132행 수집 완료
   📅 2026-04-05 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-05T00:00:00Z ~ 2026-04-06T00:00:00Z)
✅ 추출 완료! 총 18530행, 11개 컬럼 확보.
      ✅ 18530행 수집 완료
   📅 2026-04-06 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-06T00:00:00Z ~ 2026-04-07T00:00:00Z)
✅ 추출 완료! 총 35949행, 11개 컬럼 확보.
      ✅ 35949행 수집 완료
   📅 2026-04-07 데이터 요청 중...
🔍 데이터 추출 시작... (2026-04-07T00:00:00Z ~ 2026-04-08T00:00:00Z)
✅ 추출 완료! 총 35546행, 11개 컬럼 확보.
      ✅ 3

# 데이터 짜르기

# Feature Engineering

# 표준화 작업과 모델 준비

In [10]:
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from Pump_AE import PumpAutoencoder
import joblib

In [11]:
# ==========================================
# ⚖️ 4. 스케일링 및 스케일러 저장 (매우 중요!)
# ==========================================
scaler = StandardScaler()

# *중요: 스케일러는 무조건 Train 데이터로만 fit 해야 합니다! (데이터 누수 방지)
X_train_scaled = scaler.fit_transform(train_df)
X_valid_scaled = scaler.transform(valid_df) # Valid는 transform만!

# 💾 배포용 스케일러 저장 (나중에 실시간 추론 스크립트에서 불러옵니다)
scaler_path = f"{MODEL_SAVE_DIR}/scaler_{PUMP_ID}.pkl"
joblib.dump(scaler, scaler_path)
print(f"💾 스케일러 저장 완료: {scaler_path}")

# 텐서 및 데이터로더 변환
train_tensor = torch.FloatTensor(X_train_scaled)
valid_tensor = torch.FloatTensor(X_valid_scaled)

train_loader = DataLoader(TensorDataset(train_tensor, train_tensor), batch_size=64, shuffle=True)
valid_loader = DataLoader(TensorDataset(valid_tensor, valid_tensor), batch_size=64, shuffle=False)

💾 스케일러 저장 완료: ./models/scaler_P1.pkl


In [ ]:
# 피처 개수(11개)만큼 input_dim 설정
input_dim = len(feature_cols)
model = PumpAutoencoder(input_dim)

# 손실 함수(오차 계산)와 최적화 도구(학습 알고리즘) 설정
criterion = nn.MSELoss() # 평균 제곱 오차
optimizer = optim.Adam(model.parameters(), lr=0.005)

print(model)

PumpAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=16, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=4, bias=True)
    (3): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=16, bias=True)
  )
)


# 학습시작

In [12]:
# ==========================================
# 🤖 5. 모델 정의 및 학습 (Early Stopping 뼈대)
# ==========================================
input_dim = len(feature_cols)
model = PumpAutoencoder(input_dim) # (이전에 정의한 클래스 사용)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 200 # epoch을 넉넉하게 줍니다.
best_val_loss = float('inf')
model_path = f"{MODEL_SAVE_DIR}/autoencoder_{PUMP_ID}.pth"

print(f"\n🏃‍♂️ {PUMP_ID} 펌프 모델 학습 시작...")
for epoch in range(epochs):
    # --- Train Phase ---
    model.train()
    total_train_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --- Validation Phase ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in valid_loader:
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(valid_loader)
    
    # 최고 성능 갱신 시 모델 저장!
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # 💾 모델 가중치 저장
        torch.save(model.state_dict(), model_path)
        saved_flag = "⭐ (Best Model Saved!)"
    else:
        saved_flag = ""
        
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} {saved_flag}")

print(f"\n✅ 학습 완료! 최적의 모델이 저장되었습니다: {model_path}")


🏃‍♂️ P1 펌프 모델 학습 시작...
Epoch [001/200] | Train Loss: 0.3319 | Val Loss: 0.0901 ⭐ (Best Model Saved!)
Epoch [010/200] | Train Loss: 0.1355 | Val Loss: 0.0737 ⭐ (Best Model Saved!)
Epoch [020/200] | Train Loss: 0.1357 | Val Loss: 0.0735 
Epoch [030/200] | Train Loss: 0.1344 | Val Loss: 0.0732 
Epoch [040/200] | Train Loss: 0.1349 | Val Loss: 0.0732 
Epoch [050/200] | Train Loss: 0.1332 | Val Loss: 0.0742 
Epoch [060/200] | Train Loss: 0.1337 | Val Loss: 0.0741 
Epoch [070/200] | Train Loss: 0.1352 | Val Loss: 0.0753 
Epoch [080/200] | Train Loss: 0.1334 | Val Loss: 0.0759 
Epoch [090/200] | Train Loss: 0.1347 | Val Loss: 0.0735 
Epoch [100/200] | Train Loss: 0.1333 | Val Loss: 0.0741 
Epoch [110/200] | Train Loss: 0.1340 | Val Loss: 0.0735 
Epoch [120/200] | Train Loss: 0.1331 | Val Loss: 0.0726 
Epoch [130/200] | Train Loss: 0.1317 | Val Loss: 0.0741 
Epoch [140/200] | Train Loss: 0.1329 | Val Loss: 0.0733 
Epoch [150/200] | Train Loss: 0.1320 | Val Loss: 0.0735 
Epoch [160/200] | Trai